# Multimodal AI Pipeline for Breast Tumor Classification

This notebook explores how deep learning and large language models can be combined to support medical imaging analysis.

The system uses:
- **BreastMNIST dataset** (MedMNIST)
- **CNN model in PyTorch** for tumor classification
- **Qwen LLM** to generate interpretable explanations of results

Goal: explore how multimodal AI systems can improve interpretability in medical AI.

Author: Maryam Shahbaz Ali

# CNH - BreastMNIST Multimodal Pipeline (CNN + Qwen)

## 1) Environment Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-color", "-U",
                "medmnist", "tqdm", "scikit-learn", "matplotlib", "pillow", "requests"],
               check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-color", "-U",
                "transformers", "accelerate"],
               check=True)

In [ ]:
import os, sys, platform, torch
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms

import medmnist
from medmnist import INFO, Evaluator

## 2) Dataset Loading & Exploration

In [ ]:
from medmnist import BreastMNIST
train_dataset = BreastMNIST(download=True, split="train")

In [ ]:
train_dataset

In [ ]:
len(train_dataset)

In [ ]:
print(f"MedMNIST v{medmnist.__version__} @ {medmnist.HOMEPAGE}")

In [ ]:
# visualization

train_dataset.montage(length=20)

In [ ]:
train_dataset.labels

## Transformers  for Qwen


In [ ]:
# Load Qwen via Transformers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(QWEN_HF_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    QWEN_HF_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

def qwen_hf(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(qwen_hf("Write a short, clinician-friendly definition of sensitivity and specificity."))


## Train a simple CNN and evaluate Accuracy + ROC-AUC + Confusion Matrix

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from tqdm import tqdm
from medmnist import INFO
import medmnist

data_flag = "breastmnist"
info = INFO[data_flag]
DataClass = getattr(medmnist, info["python_class"])

print("Dataset:", data_flag)
print("Task:", info["task"])
print("N channels:", info["n_channels"])
print("Labels:", info["label"])

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_ds = DataClass(split="train", download=True, transform=transform)
val_ds   = DataClass(split="val",   download=True, transform=transform)
test_ds  = DataClass(split="test",  download=True, transform=transform)

BATCH_SIZE = 128
pin_mem = torch.cuda.is_available()
train_loader = data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin_mem)
val_loader   = data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_mem)
test_loader  = data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_mem)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:

# Defining a small CNN (binary classification)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)  # logits
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)

model_cnn = SmallCNN().to(device)
print(model_cnn)


In [ ]:
# Train loop + evaluation helpers
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

def eval_loader(model, loader):
    model.eval()
    all_logits = []
    all_y = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).float().view(-1)
            logits = model(x)
            all_logits.append(logits.detach().cpu())
            all_y.append(y.detach().cpu())
    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy().astype(int)

    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else float("nan")
    cm = confusion_matrix(y_true, y_pred)
    return acc, auc, cm, probs, y_true

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=1e-3)

EPOCHS = 5
best_val_auc = -1.0

for epoch in range(1, EPOCHS + 1):
    model_cnn.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for x, y in pbar:
        x = x.to(device)
        y = y.to(device).float().view(-1)
        optimizer.zero_grad()
        logits = model_cnn(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=float(loss.item()))

    val_acc, val_auc, val_cm, _, _ = eval_loader(model_cnn, val_loader)
    print(f"Epoch {epoch}: val_acc={val_acc:.4f}  val_auc={val_auc:.4f}")
    print("Val confusion matrix:\n", val_cm)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model_cnn.state_dict(), "best_cnn.pt")
        print("Saved best model to best_cnn.pt")


In [ ]:
# Final test evaluation (best checkpoint)
model_cnn.load_state_dict(torch.load("best_cnn.pt", map_location=device, weights_only=True))
test_acc, test_auc, test_cm, test_probs, test_y = eval_loader(model_cnn, test_loader)

print(f"TEST: acc={test_acc:.4f}  auc={test_auc:.4f}")
print("Test confusion matrix:\n", test_cm)


In [ ]:
# Confusion matrix visualization
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots()
im = ax.imshow(test_cm, cmap="Blues")
ax.set_title("Confusion Matrix (Test)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Malignant", "Normal/Benign"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Malignant", "Normal/Benign"])
plt.colorbar(im, ax=ax)
for (i, j), v in np.ndenumerate(test_cm):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=14,
            color="white" if test_cm[i, j] > test_cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

##  Using Qwen to write a short summary


In [ ]:
import requests

def ollama_generate(prompt, model="qwen2.5:3b-instruct"):
    r = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=300,
    )
    r.raise_for_status()
    return r.json().get("response", "")

In [ ]:
import requests

def ollama_generate(prompt, model="qwen2.5:3b-instruct"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
        },
        timeout=10  # prevents long waiting
    )
    return response.json()["response"]

In [ ]:
results_text = (
    "BreastMNIST CNN baseline results:\n"
    f"- Test Accuracy: {test_acc:.4f}\n"
    f"- Test ROC-AUC: {test_auc:.4f}\n"
    f"- Confusion Matrix: {test_cm.tolist()}\n"
)

prompt = (
    "You are assisting a medical imaging research trainee. "
    "Write a concise (120-160 words) results summary for a supervisor. "
    "Use clinician-friendly language. Mention accuracy, ROC-AUC, and interpret the confusion matrix in plain terms. "
    "Avoid hype and avoid claiming clinical deployment.\n\n"
    "Here are the results:\n"
    f"{results_text}"
)

summary = None

# Prefer Ollama if it responds; fall back to HF Transformers
try:
    summary = ollama_generate(prompt, model="qwen2.5:3b-instruct")
    print("Used: Ollama Qwen\n")
except Exception as e:
    print("Ollama unavailable, using Transformers fallback. Reason:", repr(e))
    summary = qwen_hf(prompt, max_new_tokens=220)
    print("Used: HF Qwen\n")

print(summary)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
# Qwen HF setup cell
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

def qwen_hf(prompt, max_new_tokens=220):
    messages = [
        {"role": "system", "content": "You are a precise and concise medical research writing assistant."},
        {"role": "user", "content": prompt}
    ]

    output = text_gen(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    # pipeline returns generated conversation structure
    try:
        return output[0]["generated_text"][-1]["content"]
    except Exception:
        return str(output)

In [ ]:
# Final results + summary cell

import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Exact model architecture used to save best_cnn.pt
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

# Load model
model_cnn = SmallCNN().to(device)
state_dict = torch.load("best_cnn.pt", map_location=device)
model_cnn.load_state_dict(state_dict)
model_cnn.eval()

# Evaluate on test set
def eval_loader(model, loader):
    model.eval()
    all_logits = []
    all_y = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.float().view(-1).to(device)

            logits = model(x)

            all_logits.append(logits.cpu())
            all_y.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy().astype(int)

    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, probs)
    cm = confusion_matrix(y_true, y_pred)

    return acc, auc, cm, probs, y_true

# Compute test metrics
test_acc, test_auc, test_cm, test_probs, test_y = eval_loader(model_cnn, test_loader)

print(f"TEST: acc={test_acc:.4f}  auc={test_auc:.4f}")
print("Test confusion matrix:\n", test_cm)

# BreastMNIST label mapping:
# 0 = malignant
# 1 = normal / benign
malignant_correct = int(test_cm[0, 0])
false_negatives = int(test_cm[0, 1])   # malignant predicted benign
false_positives = int(test_cm[1, 0])   # benign predicted malignant
benign_correct = int(test_cm[1, 1])

results_text = f"""
BreastMNIST CNN baseline results:
- Test Accuracy: {test_acc:.4f}
- Test ROC-AUC: {test_auc:.4f}
- Confusion Matrix: {test_cm.tolist()}
- Label mapping: 0 = malignant, 1 = normal/benign
- Malignant correctly predicted as malignant: {malignant_correct}
- Malignant misclassified as benign (false negatives): {false_negatives}
- Benign misclassified as malignant (false positives): {false_positives}
- Benign correctly predicted as benign: {benign_correct}
- Total test samples: {len(test_y)}
"""

prompt = f"""
You are assisting a medical imaging research trainee.

Write a concise 120-160 word summary of the CNN results for a research supervisor.

Requirements:
- Use clinician-friendly language
- Mention test accuracy and ROC-AUC
- Explain the confusion matrix in plain terms
- Explicitly state that malignant cases predicted as benign are false negatives
- Highlight why false negatives are important in cancer detection
- Use the exact counts provided below
- Do not invent numbers
- Avoid hype
- Do not claim clinical deployment or clinical readiness

Results:
{results_text}
"""

# LLM summary generation
summary = None

print("qwen_hf available:", "qwen_hf" in globals())
print("ollama_generate available:", "ollama_generate" in globals())

if "qwen_hf" in globals():
    print("Using qwen_hf")
    try:
        summary = qwen_hf(prompt, max_new_tokens=220)
    except Exception as e:
        print("qwen_hf failed:", e)
        summary = None

elif "ollama_generate" in globals():
    print("Using ollama_generate")
    try:
        summary = ollama_generate(prompt, model="qwen2.5:3b-instruct")
    except Exception as e:
        print("Ollama failed:", e)
        summary = None

if summary is None:
    print("Using manual fallback summary")
    summary = (
        f"On the BreastMNIST test set, the CNN achieved an accuracy of {test_acc:.4f} "
        f"and an ROC-AUC of {test_auc:.4f}. The confusion matrix was {test_cm.tolist()}, "
        f"using the label mapping 0 = malignant and 1 = normal/benign. The model correctly "
        f"identified {malignant_correct} malignant cases and {benign_correct} benign cases. "
        f"There were {false_negatives} malignant cases predicted as benign, which are false "
        f"negatives, and {false_positives} benign cases predicted as malignant. In cancer "
        f"detection, false negatives are especially important because they represent missed "
        f"malignant cases that could delay follow-up and treatment. These results suggest "
        f"that while the model shows moderate discrimination by ROC-AUC, its classification "
        f"behavior should be interpreted carefully given the error pattern on the test set."
    )

print("\nSummary:\n")
print(summary)